Sentence Segmentation. 

In [1]:
import spacy

In [2]:
nlp = spacy.load("en_core_web_sm")

In [3]:
doc = nlp(u"This is first sentence. This is second sentence. This is my last sentence.")

In [4]:
for sent in doc.sents:
    print(sent)

This is first sentence.
This is second sentence.
This is my last sentence.


doc.sents[0] - will fail because its not a list, its a generator. 

Pass it to list to make list of sentences - 

In [6]:
list(doc.sents)

[This is first sentence., This is second sentence., This is my last sentence.]

But maybe we wanna segment on coma and not full stop or something else - we use custom segmentation

In [7]:
doc = nlp(u'"Management is doing the right things; leadership is doing the right things." -Peter Drucker')

In [8]:
doc.text

'"Management is doing the right things; leadership is doing the right things." -Peter Drucker'

In [9]:
for sent in doc.sents:
    print(sent)

"Management is doing the right things; leadership is doing the right things."
-Peter Drucker


1. Add segmentation rule
2. Change the rule 

In [24]:
#1 - 
from spacy.language import Language
@Language.component("set_custom_boundaries")
def set_custom_boundaries(doc):
    for token in doc[:-1]:
        if token.text == ';':
            doc[token.i+1].is_sent_start = True
    return doc

In [26]:
nlp.add_pipe("set_custom_boundaries", before='parser')


<function __main__.set_custom_boundaries(doc)>

In [28]:
nlp.pipe_names

['tok2vec',
 'tagger',
 'set_custom_boundaries',
 'parser',
 'attribute_ruler',
 'lemmatizer',
 'ner']

In [29]:
doc4 = nlp(u'"Management is doing the right things; leadership is doing the right things." -Peter Drucker')

In [30]:
for sent in doc4.sents:
    print(sent)

"Management is doing the right things;
leadership is doing the right things."
-Peter Drucker


This added a rule. But now I wanna remove defaults and all new rule 

In [51]:
nlp = spacy.load('en_core_web_sm')

In [52]:
mystring = u"This is a sentence. This is another \n\n This is a third \n sentence." 

In [53]:
print(mystring)

This is a sentence. This is another 

 This is a third 
 sentence.


In [54]:
doc = nlp(mystring)

In [55]:
for sent in doc.sents:
    print(sent)

This is a sentence.
This is another 

 This is a third 
 sentence.


But I want new line as a new sentence 

In [56]:
from spacy.pipeline import Sentencizer

In [57]:
@Language.component("split_on_newlines")
def split_on_newlines(doc):
    start = 0
    seen_newline = False
    for word in doc:
        if seen_newline :
            yield doc[start:word.i]
            start = word.i
            seen_newline = False
        elif word.text.startswith("\n"):
            seen_newline = True
    return doc

In [58]:
nlp.add_pipe("split_on_newlines" , before="parser")
doc = nlp(mystring)
for sentence in doc.sents:
    print(sentence)

ValueError: [E005] Pipeline component 'split_on_newlines' returned <class 'generator'> instead of a Doc. If you're using a custom component, maybe you forgot to return the processed Doc?